# Audio and image embedding space visualization

The notebook loads SpokenCOCO audio-image pairs, encodes both modalities using the trained SAIL modeland visualizes their embeddings in a shared space using PCA and t-SNE.

In [4]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").exists():
    raise FileNotFoundError("Start from the root of the project")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [5]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Subset

from models.contrastive_model import ContrastiveModel
from torch_datasets.spoken_coco_dataset import SpokenCocoDataset
from torch_datasets.transforms import Audio_ValTransform, Img_ValTransform
from training.checkpointing import load_checkpoint
from training.configs.baseline_config import Config

In [6]:
config = Config()
DATA_DIR = (project_root / config.data_dir).resolve()
CHECKPOINT_PATH = (project_root / config.checkpoint_dir / "final_model_best.pth").resolve()
N = 100
SEED = 42

device = torch.device(config.device)
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")
if not (DATA_DIR / "SpokenCOCO_val.json").is_file():
    raise FileNotFoundError(f"Annotations not found: {DATA_DIR / 'SpokenCOCO_val.json'}")

In [7]:
dataset = SpokenCocoDataset(
    img_dir=DATA_DIR / "val2014",
    audio_dir=DATA_DIR / "wavs" / "val",
    ann_file=DATA_DIR / "SpokenCOCO_val.json",
    img_transform=Img_ValTransform(size=config.size, use_ccrop=config.use_ccrop),
    audio_transform=Audio_ValTransform(
        sample_rate=config.sample_rate,
        normalize=config.normalize,
    ),
    sample_rate=config.sample_rate,
)

sample_size = min(N, len(dataset))
if sample_size < 2:
    raise ValueError(f"Need at least 2 pairs, found {sample_size}.")
indices = random.Random(SEED).sample(range(len(dataset)), sample_size)
loader = DataLoader(Subset(dataset, indices), batch_size=16, shuffle=False, num_workers=0)
print(f"img-audio pairs: {sample_size}")

img-audio pairs: 100


In [8]:
model = ContrastiveModel(
    visual_model_name=config.visual_model_name,
    audio_model_name=config.audio_model_name,
    shared_dim=config.shared_dim,
    projection_type=config.projection_type,
    dropout=config.dropout,
    freeze_visual_backbone=config.freeze_visual_backbone,
    freeze_audio_backbone=config.freeze_audio_backbone,
)
epoch, best_val_loss = load_checkpoint(CHECKPOINT_PATH, model)
model.to(device).eval()
print(f"Loaded {CHECKPOINT_PATH.name}: epoch {epoch}, val_loss {best_val_loss:.4f}")

Loading weights: 100%|██████████| 49/49 [00:00<00:00, 21079.07it/s]


Loaded final_model_best.pth: epoch 6, val_loss 0.5196


In [ ]:
audio_embeddings = []
image_embeddings = []

with torch.inference_mode():
    for images, audio, lengths, _, _ in loader:
         images = images.to(device)
         audio = audio.to(device)
         lengths = lengths.to(device)
         attention_mask = torch.arange(audio.shape[1], device=device)[None, :] < lengths[:, None]

         audio_embeddings.append(model.encode_audio(audio, attention_mask, normalize=True).cpu())
         image_embeddings.append(model.encode_visual(images, normalize=True).cpu())

    audio_embeddings = torch.cat(audio_embeddings).numpy()
    image_embeddings = torch.cat(image_embeddings).numpy()
    combined = np.vstack([image_embeddings, audio_embeddings])
    n_components = min(50, combined.shape[0], combined.shape[1])
    combined_pca = PCA(n_components=n_components, svd_solver="full").fit_transform(combined)
    tsne = TSNE(n_components=2, random_state=SEED, init="pca", learning_rate="auto")
    combined_tsne = tsne.fit_transform(combined_pca)
    image_tsne = combined_tsne[:sample_size]
    audio_tsne = combined_tsne[sample_size:]

In [ ]:
explained = PCA(n_components=n_components, svd_solver="full").fit(combined).explained_variance_ratio_.sum()
print(f"Variation explained by PCA: {explained:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
ax.scatter(image_tsne[:, 0], image_tsne[:, 1], alpha=0.8, label="Obraz")
ax.scatter(audio_tsne[:, 0], audio_tsne[:, 1], alpha=0.8, label="Audio")
for i in range(sample_size):
    ax.plot(
        [image_tsne[i, 0], audio_tsne[i, 0]],
        [image_tsne[i, 1], audio_tsne[i, 1]],
        color="gray",
        alpha=0.25,
    )
ax.set_title("SpokenCOCO embedding: image vs audio")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend()
plt.show()